#### Use below code to convert the csv files to an sqlite database dump.

In [ ]:
import pandas as pd
import sqlite3
import glob
import os

In [ ]:
eventTypeTableFilenames = [fn for fn in glob.glob('event_*.csv') 
                           if not fn == "event_map_type.csv" and not fn == "event_object.csv"]
objectTypeTableFilenames =  [fn for fn in glob.glob('object_*.csv') 
                             if not fn == "object_map_type.csv" and not fn == "object_object.csv"]

TABLES = dict()

TABLES["event"] = pd.read_csv("event.csv", sep=";")
TABLES["event_map_type"] = pd.read_csv("event_map_type.csv", sep=";")
TABLES["event_object"] = pd.read_csv("event_object.csv", sep=";").drop_duplicates()
TABLES["object"] = pd.read_csv("object.csv", sep=";")
TABLES["object_object"] = pd.read_csv("object_object.csv", sep=";")
TABLES["object_map_type"] = pd.read_csv("object_map_type.csv", sep=";")

for fn in eventTypeTableFilenames:
    table_name = fn.split(".")[0]
    table = pd.read_csv(fn, sep=";")
    table["ocel_time"] = pd.to_datetime(
        table["ocel_time"], format="ISO8601"
    ).apply(
        lambda x: x.isoformat(timespec='milliseconds') + "Z"
    )
    TABLES[table_name] = table
    
for fn in objectTypeTableFilenames:
    table_name = fn.split(".")[0]
    table = pd.read_csv(fn, sep=";")
    TABLES[table_name] = table

In [ ]:
sql_path = "congestion.sqlite"
if os.path.exists(sql_path):
    os.remove(sql_path)

In [ ]:
conn = sqlite3.connect(sql_path)
for tn, df in TABLES.items():
    df.to_sql(tn, conn, index=False)
conn.close()

In [ ]:
import pm4py
ocel = pm4py.read_ocel2_sqlite("congestion.sqlite")
ocel

In [ ]:
pm4py.write_ocel2_json(ocel,'congestion.json')

In [ ]:
ocel.get_extended_table()